> **Not executed by the repo author: run in Colab or in your course lab; versions may need updates.**
> Package pins below were chosen from memory of releases that existed; if a pin fails, install without the version and note what you got.

# Lab 7: Attacking an NLP model with TextAttack

**Goal:** run real TextAttack recipes (TextFooler, DeepWordBug) against a small Hugging Face sentiment model and measure attack success.

- **Runtime:** Colab, T4 GPU recommended.
- **Time:** about 20 minutes.
- **Offline twin:** `../labs/07-textattack-adversarial/` builds a greedy word-swap attack and a normalization defense in pure Python.
- **Safety note:** attack only models you own or are allowed to test. TextAttack may conflict with the newest numpy/transformers on Colab; if the install fails, use a fresh runtime and install `textattack` without a version.

In [ ]:
!pip install -q textattack==0.3.10

## Step 1: Load a victim model
`textattack/distilbert-base-uncased-SST-2` is a small sentiment model published by the TextAttack authors (check the Hub if the name has changed).

In [ ]:
import transformers, textattack
name = "textattack/distilbert-base-uncased-SST-2"
model = transformers.AutoModelForSequenceClassification.from_pretrained(name)
tokenizer = transformers.AutoTokenizer.from_pretrained(name)
wrapper = textattack.models.wrappers.HuggingFaceModelWrapper(model, tokenizer)

## Step 2: Build an attack from a recipe
A TextAttack attack = goal function + constraints + transformation + search method. TextFooler swaps words for synonyms; DeepWordBug makes character typos. These are the same two ideas as the offline lab.

In [ ]:
from textattack.attack_recipes import TextFoolerJin2019, DeepWordBugGao2018
from textattack.datasets import HuggingFaceDataset
dataset = HuggingFaceDataset("glue", "sst2", "validation", shuffle=True)

In [ ]:
from textattack import Attacker, AttackArgs
attack = TextFoolerJin2019.build(wrapper)
results = Attacker(attack, dataset, AttackArgs(num_examples=20, disable_stdout=False)).attack_dataset()

## Step 3: Try the typo attack and compare

In [ ]:
attack2 = DeepWordBugGao2018.build(wrapper)
results2 = Attacker(attack2, dataset, AttackArgs(num_examples=20)).attack_dataset()

## Step 4: Read the summary table
Note the **attack success rate**, the **average percent of words perturbed**, and the number of queries. Write them next to the offline lab numbers.

## Step 5: Defense ideas to test
- Add typo correction before the classifier (like `normalize()` in the offline lab) and rerun the attack.
- Adversarial training: `textattack augment` or attack your own training data and add the results.
- Remember an adaptive attacker will search for what your defense misses.